# PaliGemma 2: Türkçe fişlerde 224 / 448 ve İngilizce / Türkçe soru
**Tuğçe Karakuş · Araştırma ve portföy projesi**

Bu Colab defteri iki ayrı veri yolunu destekler. Varsayılan **sentetik pilot**, 50 fiş görselini ve doğru toplamlarını üretir; kendi fişlerinizi yüklemeniz gerekmez. Bu programatik görseller gerçek fiş veya telefon fotoğrafı değildir ve sonuçları gerçek fiş başarımı olarak yorumlanamaz. Gerçek fiş deneyi için 30–50 farklı fiş ve insan tarafından doğrulanmış toplam gerekir.

Önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin. [mix-224](https://huggingface.co/google/paligemma2-3b-mix-224) ve [mix-448](https://huggingface.co/google/paligemma2-3b-mix-448) sayfalarında Google erişim koşullarını kendi hesabınızla kabul edin. Giriş token’ını paylaşmayın.

Dört koşul: aynı görseller, iki model ve aynı toplamı isteyen İngilizce/Türkçe soru. Sonuç dosyaları seçilen veri türünü açıkça kaydeder.

In [ ]:
%pip -q install transformers==4.57.1 accelerate==1.11.0 huggingface_hub==0.36.0 sentencepiece==0.2.1 'Pillow>=10.4,<13' 'numpy>=1.26,<3' 'matplotlib>=3.8,<4'

## 1. Projeyi GitHub’dan alın
Bu hücre projeyi indirir ve temel kontrolleri çalıştırır. README ve veri yönergesini gözden geçirin.

In [ ]:
import sys, json, zipfile, io, subprocess
from pathlib import Path
from google.colab import files
from IPython.display import display, Image as DisplayImage
BASE = Path('/content/paligemma2_portfolio')
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / 'paligemma2-turkish-receipts'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/karakustugce/paligemma2-turkish-receipts.git', str(PROJECT)], check=True)
assert (PROJECT/'pg2receipts/run.py').is_file(), 'Proje indirilemedi.'
sys.path.insert(0, str(PROJECT))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=PROJECT, check=True)
print('Proje ve temel kontroller hazır.')

## 2. Model erişimi ve GPU
CUDA GPU bulunmuyorsa burada durun ve çalışma zamanını değiştirin. Modeller sırayla ve FP16 ile yüklenir. GPU belleği veya erişim sorunu olduğunda koşu durur; bunu başarısız model cevabı olarak saymayın.

In [ ]:
import torch
from huggingface_hub import notebook_login
assert torch.cuda.is_available(), 'GPU çalışma zamanı seçilmedi.'
print('GPU:', torch.cuda.get_device_name(0))
print('Bellek (GB):', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))
notebook_login()

## 3. Veri yolunu seçin

Aşağıdaki hücrede `DATASET_MODE = 'synthetic'` varsayılandır. Üretici fiş benzeri PNG görselleri, toplam etiketlerini ve veri notunu çalışma anında oluşturur; 50 görsel ve etiket için manuel giriş gerekmez. İlk üç örnek görseli inceleyip sonra veri setini dondurun.

Gerçek fiş deneyi için `DATASET_MODE` değerini `'real_sample'` yapıp özel referans CSV’yi yükleyin veya `'own_real'` yapıp doldurulmuş `manifest.csv` ile fiş görüntülerini/ZIP’ini yükleyin. Gerçek fişlerde her nihai toplamı görüntüden doğrulayın. Gerçek ve sentetik kayıtlar tek koşuda karıştırılamaz.

In [ ]:
import csv
from PIL import Image, ImageOps
from pg2receipts.core import load_manifest, parse_amount

DATASET_MODE = 'synthetic'  # 'synthetic', 'real_sample', or 'own_real'

if DATASET_MODE == 'synthetic':
    DATA = BASE/'synthetic_pilot_data'
    subprocess.run([
        sys.executable, '-m', 'pg2receipts.synthetic',
        '--out', str(DATA), '--count', '50', '--seed', '2026'
    ], cwd=PROJECT, check=True)
    MANIFEST = DATA/'manifest.csv'
    IMAGES = DATA/'images'

elif DATASET_MODE == 'real_sample':
    DATA = BASE/'private_real_data'
    IMAGES = DATA/'images'
    IMAGES.mkdir(parents=True, exist_ok=True)
    MANIFEST = DATA/'manifest.csv'
    print('Özel kaynak referans CSV dosyasını yükleyin; bu dosya açık depoda bulunmuyor.')
    reference_upload = files.upload()
    assert len(reference_upload) == 1 and next(iter(reference_upload)).lower().endswith('.csv')
    draft_input = DATA/'source_reference_private.csv'
    draft_input.write_bytes(next(iter(reference_upload.values())))
    subprocess.run([
        sys.executable, '-m', 'pg2receipts.prepare',
        '--out', str(DATA), '--draft-manifest', str(draft_input)
    ], cwd=PROJECT, check=True)
    source_csv = MANIFEST if MANIFEST.exists() else DATA/'manifest_draft.csv'
    with source_csv.open(encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        if row.get('annotation_status') == 'human_verified':
            continue
        with Image.open(IMAGES/row['image_path']) as raw:
            preview = ImageOps.exif_transpose(raw).convert('RGB')
            preview.thumbnail((600, 1100))
            display(preview)
        answer = input(
            f"{row['doc_id']} — önerilen toplam {row['true_total']}; "
            "doğruysa Enter, değilse doğru toplam: "
        ).strip()
        value = parse_amount(answer or row['true_total'])
        if value is None:
            raise ValueError('Tutar belirsiz. Geçerli iki ondalıklı tutar girin.')
        row['true_total'] = str(value)
        row['annotation_status'] = 'human_verified'
        with MANIFEST.open('w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)

elif DATASET_MODE == 'own_real':
    DATA = BASE/'private_real_data'
    IMAGES = DATA/'images'
    IMAGES.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    manifest_bytes = None
    image_names = set()

    def save_image(name, data):
        filename = Path(name).name
        assert filename not in image_names, 'Tekrarlanan görüntü adı: '+filename
        image_names.add(filename)
        (IMAGES/filename).write_bytes(data)

    for name, data in uploaded.items():
        suffix = Path(name).suffix.lower()
        if suffix == '.csv':
            assert manifest_bytes is None, 'Tek bir manifest CSV yükleyin.'
            manifest_bytes = data
        elif suffix == '.zip':
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for item in z.infolist():
                    assert '..' not in Path(item.filename).parts and not Path(item.filename).is_absolute()
                    if not item.is_dir() and Path(item.filename).suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
                        save_image(item.filename, z.read(item))
        elif suffix in {'.jpg', '.jpeg', '.png', '.webp'}:
            save_image(name, data)
        else:
            raise ValueError('Desteklenmeyen dosya: '+name)
    assert manifest_bytes is not None, 'Doğrulanmış manifest.csv dosyasını da yükleyin.'
    MANIFEST = DATA/'manifest.csv'
    MANIFEST.write_bytes(manifest_bytes)
    print('Gerçek fiş toplamlarını model koşusundan önce görüntülerden doğrulayın.')

else:
    raise ValueError("DATASET_MODE 'synthetic', 'real_sample' veya 'own_real' olmalı.")

docs = load_manifest(MANIFEST, IMAGES)
print('Geçerli farklı belgeler:', len(docs))
if DATASET_MODE == 'synthetic':
    for doc in docs[:3]:
        with Image.open(doc['image']) as preview:
            display(preview.copy())
    print('Etiketler aynı sentetik işlem kaydından otomatik hesaplandı.')
assert input('Veri yolunu ve örnekleri kontrol ettim, bu veri setini donduruyorum. EVET: ').strip().upper() == 'EVET'

## 4. Aynı belgeleri iki modele okutun

Model indirme ilk çalıştırmada zaman alır. Modeller sırayla, FP16 ile yüklenir. Koşu görüntü hash’lerini, toplamları ve veri türünü kaydeder. Kesilen koşuyu aynı veri ve ortamla yeniden çalıştırmak kaldığı yerden devam eder; değişen veri veya yazılım sürümü için yeni çıktı klasörü gerekir.

Gerçek fiş toplamları modele verilmez. Sentetik toplamlar üreticiden gelir; rapor bu veri türünü açıkça belirtir ve gerçek kullanım ya da ekonomik kazanç iddiasına izin vermez.

In [ ]:
OUT = BASE/f'{DATASET_MODE}_run'
command = [
    sys.executable, '-m', 'pg2receipts.run',
    '--manifest', str(MANIFEST), '--images', str(IMAGES), '--out', str(OUT)
]
if (OUT/'run_config.json').exists():
    command.append('--resume')
subprocess.run(command, cwd=PROJECT, check=True)

## 5. Analiz ve görseller

Başarı, toplam tutarın tam eşleşmesidir. Açıklamalı veya birden fazla tutarlı cevaplar başarısız sayılır; biçim hataları ayrıca gösterilir. Üretim süresi ve GPU belleği ölçülür; insan kontrol süresi ölçülmez.

Sentetik pilotta rapor ve doğruluk/gpu grafiği veri türünü açıkça yazar. Sentetik sonuçlardan gerçek fiş performansı veya ekonomik tasarruf çıkarılmayacağı için ekonomik senaryo grafiği üretilmez. Gerçek fiş yolundaki küçük örneklem de bütün Türkçe belgeleri temsil etmez.

In [ ]:
subprocess.run([sys.executable, '-m', 'pg2receipts.analyze', '--run', str(OUT)], cwd=PROJECT, check=True)
summary = json.loads((OUT/'summary.json').read_text())
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(OUT/'accuracy_and_resources.png')))
economic_chart = OUT/'economic_scenario.png'
if economic_chart.exists():
    display(DisplayImage(filename=str(economic_chart)))
else:
    print('Ekonomik senaryo grafiği sentetik veri için özellikle üretilmedi.')
print((OUT/'report.md').read_text())

## 6. Sonuç paketini indirin

Paket ham yanıtları, referans toplamları, sürümleri, özeti, raporu ve grafikleri içerir; fiş görüntülerini veya giriş token’ını içermez. Sentetik sonuç paketini de gerçek fiş sonucu gibi sunmayın. Kamuya açık depoya eklemeden önce ham yanıtları, referansları ve veri türünü gözden geçirin.

In [ ]:
archive = BASE/f'PaliGemma_2_{DATASET_MODE}_Sonuclari.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            z.write(path, arcname=path.name)
files.download(str(archive))